# RAG-система с ChromaDB

## Архитектура

Фильтруем по категории -> забираем описание и три наиболее близкие по косинусной близости категории жалоб -> описание и категории кладутся в промпт для LLM

## Документы в ChromaDB

| Тип | Количество | Размер | Роль |
|-----|--------|--------|-----------|
| product_description | 1 на продукт | 150–300 слов | Контекст для понимания продукта |
| category | 5–8 на продукт | 50–80 слов | Темы для сопоставления с существующими и определением новых |

## Три версии разметки, сравниваемые в рамках эксперимента

| Версии | Контекст в промпте |
|-------|--------------------|
| no_context | Отсутствует |
| static | Список названий категорий |
| rag | Описание продукта и топ-3 релевантные категории с описаниями |

In [6]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

In [7]:
BASE_DIR = Path("..").resolve()
DATA_DIR = BASE_DIR / "results/notebooks"
CHROMA_DIR = BASE_DIR / "data" / "chroma_db"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
OLLAMA_BASE_URL = "http://localhost:11434"
MODEL_NAME = "qwen2.5:7b-instruct-q4_K_M"
GENERATION_CONFIG = {
    "temperature": 0.0,
    "seed": 42,
    "top_p": 0.9,
    "num_predict": 300,
}

In [9]:
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
TOP_K_CATEGORIES = 3

PRODUCT_ID_MAP = {
    "Видео-платформа": "video_platform",
    "Маркетплейс": "internet_shop",
}

In [ ]:
def check_ollama_health():
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        return r.status_code == 200
    except Exception:
        return False

def call_ollama(user_message: str, system_prompt: str | None = None, model: str = MODEL_NAME, config: dict = GENERATION_CONFIG):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": user_message})
    t0 = time.time()
    r = requests.post(
        f"{OLLAMA_BASE_URL}/api/chat",
        json={"model": model, "messages": messages, "stream": False, "options": config},
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    return {
        "content": data["message"]["content"],
        "elapsed": round(time.time() - t0, 2),
        "tokens":  data.get("eval_count", 0),
    }

def parse_json_response(text: str) -> dict | None:
    try:
        return json.loads(text.strip())
    except json.JSONDecodeError:
        pass
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    m = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return None

def build_cluster_info(df: pd.DataFrame, topic_id: int, keywords_dict: dict, n_comments: int | None = None):
    keywords   = keywords_dict.get(str(topic_id), [])
    cluster_df = df[df["topic"] == topic_id]
    size = len(cluster_df)
    if n_comments is None:
        n_comments = min(15, max(5, size // 15))
    comments = (
        cluster_df["text"].dropna()
        .sample(min(n_comments, size), random_state=42)
        .tolist()
    )
    product_name = cluster_df["product"].iloc[0] if size > 0 else ""
    return {
        "topic_id": topic_id,
        "keywords": keywords,
        "comments": comments,
        "size": size,
        "product": product_name,
        "product_id": PRODUCT_ID_MAP.get(product_name, "unknown"),
    }

print(f"Ollama {'запущен' if check_ollama_health() else 'недоступен'}")

Ollama запущен


---
## RAG-документы

- `PRODUCT_DOCS` — 1 документ на продукт, 150–300 слов с пояснениями, что это за сервис, его ключевые функции, типичная аудитория, модели доступа.
- `CATEGORIES_DOCS` — по одному документу на каждую известную категорию жалоб, 50–80 слов. Название, признаки и типичные формулировки клиентов.

In [16]:
PRODUCT_DOCS = {
    "video_platform": {
        "name": "Видео-платформа",
        "description": """Видео-платформа — это онлайн-кинотеатр и ТВ-сервис, где пользователь
            заходит в приложение или на сайт, входит по номеру телефона, выбирает
            профиль и дальше ищет, что посмотреть: фильм, сериал, телеканал,
            детский контент или конкретный тайтл через поиск. На главном экране
            пользователь листает подборки и карточки контента, открывает каталог
            с разделами вроде «Фильмы», «Сериалы» и «Телеканалы», переходит в карточку
            и запускает просмотр в плеере. Внутри сценария просмотра важны точки, где
            часто возникает трение: вход по SMS-коду, выбор профиля, ввод ПИН-кода, поиск
            нужного фильма или канала, запуск серии, перемотка, пауза, продолжение просмотра,
            переключение между сериями и выбор качества или версии воспроизведения.
            Сервис отдельно выделяет темы подписки, оплаты, управления подпиской, промокодов,
            родительского контроля, профилей и ПИН-кода, а также исправления сбоев в работе приложения.
            Это значит, что жалобы могут касаться не только самого видео, но и списаний, доступа к контенту,
            активации промокода, ограничений по профилю, настроек и нестабильной работы приложения.
            Также для части сценариев важны установка приложения, обновление, скачивание
            и запуск на телевизоре или приставке.

            Видео-платформа работает на смартфонах и планшетах, в веб-версии на компьютере,
            на телевизорах и приставках. В официальной поддержке указаны Android TV,
            Apple TV, Smart TV на LG WebOS и Samsung Tizen, отдельные модели Hisense,
            Toshiba, KIVI, устройства на Яндекс ТВ и часть устройств SberDevices;
            для некоторых старых или слабых устройств прямо отмечается
            возможная нестабильная работа.""",
    },
    "internet_shop": {
        "name": "Маркетплейс МТС",
        "description": """Маркетплейс МТС — это сайт, где пользователь ищет и покупает
            технику и сопутствующие товары: смартфоны, планшеты, ноутбуки, умные часы,
            наушники, колонки, устройства для умного дома, роутеры, ТВ-товары, аксессуары,
            SIM-карты, услуги и софт. Пользователь заходит в каталог, открывает карточки товаров,
            смотрит цену, характеристики, отзывы, наличие, варианты доставки и самовывоза,
            добавляет товар в корзину и оформляет заказ. На уровне пользовательского опыта
            важны поиск по каталогу, фильтры, сравнение моделей, выбор цвета и памяти, проверка
            наличия в нужном городе или магазине, а также переходы между каталогом, карточкой
            товара и корзиной. Типичные точки трения в таком сервисе: войти в аккаунт
            или оформить заказ без лишних шагов; применить кешбэк или понять, почему он не списывается;
            выбрать способ оплаты; оформить рассрочку или кредит; заполнить анкету; дождаться одобрения;
            выбрать доставку или самовывоз; отследить статус заказа; получить уведомление о подтверждении,
            переносе или отмене; разобраться с обменом, возвратом и ремонтом.
            Отдельный сценарий связан с рассрочкой: на сайте нужно добавить товар в корзину,
            выбрать самовывоз и «рассрочку онлайн», заполнить анкету, дождаться решения банка
            и затем прийти в салон подписать договор; при этом рассрочка может быть
            недоступна для курьерской доставки. Для части заказов важны и сроки
            начисления кешбэка после получения товара.

            Сервис работает как веб-маркетплейс МТС на сайте shop.mts.ru.
            Пользователь обычно заходит в него с телефона или компьютера через браузер,
            оформляет заказ онлайн, а получает товар курьером или самовывозом
            из салона-магазина МТС.""",
    },
}

In [17]:
CATEGORIES_DOCS = {
    "video_platform": [
        {
            "id": "video_platform_auth",
            "name": "Проблемы со входом в приложение",
            "description": """Жалобы на невозможность войти в аккаунт и открыть сервис
                после запуска приложения. У пользователя не получается авторизоваться
                по номеру телефона, коду, паролю или через МТС ID: экран входа зависает,
                код не приходит, кнопка не срабатывает, вход сбрасывается, появляется ошибка,
                приложение снова просит войти. Типичные формулировки: «не могу войти в
                приложение», «не заходит в Видео-платформа», «не приходит код», «ошибка при входе»,
                «вылетает при авторизации», «снова просит войти», «не входит по номеру».""",
        },
        {
            "id": "video_platform_playback",
            "name": "Проблемы с воспроизведением или запуском фильмов/каналов",
            "description": """Проблема связана с тем, что в Видео-платформа не запускается или
                некорректно воспроизводится фильм, сериал, телеканал или видео в плеере.
                У пользователя не начинается просмотр, экран зависает, идет бесконечная
                загрузка, появляется черный экран, выбивает из плеера, видео останавливается,
                лагает, тормозит, прерывается или канал не открывается. Формулировки: «не запускается фильм»,
                «канал не включается», «вечная загрузка», «черный экран при просмотре»,
                «видео зависает», «плеер вылетает», «не могу посмотреть сериал».""",
        },
        {
            "id": "video_platform_interface",
            "name": "Неудобный интерфейс/плеер",
            "description": """Жалобы на неудобный интерфейс и плеер Видео-платформа связаны с тем,
                что пользователю трудно ориентироваться в приложении и управлять просмотром.
                Неудобно искать фильм, переключать серии, перематывать, ставить на паузу, выбирать 
                сезон, включать субтитры или менять звук; кнопки, меню, экран плеера и карточки контента
                кажутся непонятными, перегруженными или нелогичными. «Неудобный плеер», «ничего не найти»,
                «кнопки расположены ужасно», «перемотка неудобная», «сложный интерфейс», «непонятное меню»,
                «неудобно переключать серии».""",
        },
        {
            "id": "video_platform_choice",
            "name": "Ассортимент фильмов/каналов",
            "description": """Проблема связана с ассортиментом контента в Видео-платформа: пользователю
                не хватает фильмов, сериалов, ТВ-каналов, детского или тематического контента,
                нужные позиции отсутствуют в каталоге или выбор кажется слишком бедным.
                Пользователь ищет конкретный фильм, канал, жанр, подборку, но не находит нужное,
                видит мало новинок, мало популярных тайтлов или ограниченный список доступного.
                «Мало фильмов», «нечего смотреть», «нет нужного канала», «бедный выбор», «нет новинок»,
                «нет моего фильма», «слишком мало контента».""",
        },
        {
            "id": "video_platform_glitches",
            "name": "При переходах между разделами приложение тормозит",
            "description": """Суть жалобы — приложение Видео-платформа тормозит при навигации между разделами,
            экранами и карточками контента: пользователь открывает главную, каталог, поиск,
            профиль, ТВ-каналы или подборки, а переходы идут с задержкой, страницы долго грузятся,
            интерфейс подвисает, скролл дергается, нажатия срабатывают не сразу, экран может
            ненадолго замирать. Типичные формулировки: «приложение тормозит», «долго грузятся разделы»,
            «все виснет при переходах», «медленно открывается каталог», «лагает интерфейс»,
            «тормозит между вкладками», «страницы открываются с задержкой».""",
        },
        {
            "id": "video_platform_advert",
            "name": "Навязчивая реклама",
            "description": """Пользователь жалуется на навязчивую рекламу в Видео-платформа: реклама
            слишком часто показывается в приложении и плеере, мешает просмотру фильмов,
            сериалов и каналов, отвлекает на экране, прерывает запуск или воспроизведение
            видео, всплывает перед контентом, во время просмотра или при переходах по разделам.
            Типичные формулировки: «слишком много рекламы», «реклама мешает смотреть»,
            «постоянно выскакивает реклама», «реклама перед каждым видео», «нельзя пропустить
            рекламу», «навязали рекламу», «одна реклама везде».""",
        },
        {
            "id": "video_platform_subscription",
            "name": "Дорогая подписка, сложно отключить подписку",
            "description": """Пользователь жалуется на высокую цену подписки Видео-платформа и трудности
            с ее отключением: подписка кажется дорогой, списания кажутся неожиданными, непонятно,
            где отменить автопродление, как отключить пакет, тариф или пробный период.
            На экране сложно найти нужный раздел, кнопка отключения неочевидна, условия
            списания вызывают недовольство. Типичные формулировки: «дорогая подписка»,
            «слишком дорого», «не могу отключить подписку», «сложно отменить автопродление»,
            «где отключить Видео-платформа», «непонятное списание», «не отключается подписка».""",
        },
    ],
    "internet_shop": [
        {
            "id": "shop_website",
            "name": "Плохо работает сайт, медленно загружается",
            "description": """Жалобы связаны с тем, что сайт маркетплейса МТС работает
            нестабильно или слишком медленно: страницы долго открываются, каталог и карточки
            товаров грузятся с задержкой, поиск, корзина и оформление заказа подвисают, кнопки
            нажимаются не сразу, сайт может зависать, обновляться с ошибками или вовсе не загружаться.
            Типичные формулировки: «сайт тормозит», «страница долго грузится», «ничего не открывается»,
            «сайт зависает», «медленно работает магазин», «не грузится каталог»,
            «невозможно оформить заказ».""",
        },
        {
            "id": "shop_interface",
            "name": "Много отвлекающих элементов на сайте",
            "description": """Жалобы связаны с тем, что в маркетплейсе МТС на сайте
            слишком много баннеров, всплывающих окон, рекламных блоков, плашек и лишних элементов,
            которые отвлекают от выбора товара и оформления заказа. Пользователю мешают яркие блоки,
            навязчивые предложения, перекрывающие экран окна, сложно найти нужную кнопку, фильтр,
            описание или корзину. «Слишком много рекламы», «мешают баннеры», «всплывают окна»,
            «сайт перегружен», «много лишнего на экране», «неудобно искать товар»,
            «отвлекающие элементы».""",
        },
        {
            "id": "shop_assortment",
            "name": "Не нашел нужный товар",
            "description": """Пользовательская жалоба связана с тем, что в маркетплейсе МТС
            не удаётся найти нужный товар, модель, версию или аксессуар в каталоге, поиске
            или нужной категории. Пользователь ищет по названию, бренду, фильтрам, разделам,
            но нужная позиция не отображается, отсутствует в выдаче, карточках или списке товаров.
            «Не нашел нужный товар», «нет нужной модели», «в поиске ничего нет»,
            «не могу найти аксессуар», «нет в каталоге», «не отображается товар», «не нашел нужный телефон».""",
        },
        {
            "id": "shop_price",
            "name": "Не устроила цена на товар",
            "description": """Жалоба связана с тем, что пользователя не устраивает цена
            товара в маркетплейсе: стоимость кажется слишком высокой, цена на сайте
            не совпадает с ожиданиями, покупка воспринимается как невыгодная, из-за цены
            пользователь отказывается от оформления заказа или откладывает покупку. В отзыве
            обычно упоминают цену, стоимость, дороговизну, скидку, акцию, итоговую сумму:
            «слишком дорого», «цена завышена», «не устроила стоимость», «дороже чем ожидал»,
            «нет нормальной скидки», «товар неоправданно дорогой», «невыгодная цена».""",
        },
        {
            "id": "shop_make_order",
            "name": "Сложный процесс оформления заказа",
            "description": """Суть жалобы — оформить заказ в маркетплейсе МТС сложно,
            долго и неудобно: много шагов, экранов, полей и подтверждений. Пользователь не может
            быстро перейти от корзины к оплате, путается в форме, заново вводит данные,
            не сразу понимает, как выбрать доставку, магазин, способ оплаты или
            применить промокод. Типичные формулировки: «слишком долго оформлять заказ»,
            «слишком много шагов», «непонятное оформление», «запутанная корзина»,
            «не могу оформить заказ», «постоянно что-то заполнять», «сложная оплата».""",
        },
        {
            "id": "shop_product_condition",
            "name": "Не устроило состояние товара при получении",
            "description": """Пользователь жалуется на состояние товара при получении:
            заказ приходит повреждённым, грязным, мятым, вскрытым, с царапинами, сколами,
            потёртостями, трещинами или другими видимыми дефектами, из-за чего товар выглядит б/у,
            неаккуратным или непригодным для нормального использования. Проблема проявляется
            при осмотре коробки, упаковки, корпуса, экрана, комплектующих. Типичные формулировки:
            «товар пришёл повреждённый», «упаковка вскрыта», «коробка мятая», «весь в царапинах»,
            «как будто б/у», «грязный товар», «есть сколы и трещины».""",
        },
    ],
}

## Индексация документов

In [18]:
def setup_chromadb(chroma_dir: Path, embedding_model: str, force_recreate: bool = True):
    client = chromadb.PersistentClient(path=str(chroma_dir))
    ef = SentenceTransformerEmbeddingFunction(model_name=embedding_model)

    if force_recreate:
        try:
            client.delete_collection("rag_product_context")
            print("Старая коллекция 'rag_product_context' удалена.")
        except Exception:
            pass

    collection = client.get_or_create_collection(
        name="rag_product_context",
        embedding_function=ef,
        metadata={"hnsw:space": "cosine"},
    )
    return client, collection


def load_documents(collection, product_docs: dict, categories_docs: dict):
    documents, ids, metadatas = [], [], []

    for pid, prod in product_docs.items():
        documents.append(prod["description"])
        ids.append(f"{pid}_description")
        metadatas.append({
            "product_id": pid,
            "type":       "product_description",
            "name":       prod["name"],
        })

    for pid, cats in categories_docs.items():
        for cat in cats:
            full_text = cat["name"] + "\n" + cat["description"]
            documents.append(full_text)
            ids.append(cat["id"])
            metadatas.append({
                "product_id":  pid,
                "type":        "category",
                "name":        cat["name"],
                "category_id": cat["id"],
            })

    collection.add(documents=documents, ids=ids, metadatas=metadatas)
    return len(documents)

In [19]:
chroma_client, rag_collection = setup_chromadb(
    CHROMA_DIR, EMBEDDING_MODEL
)

n_docs = load_documents(rag_collection, PRODUCT_DOCS, CATEGORIES_DOCS)
print(f"Проиндексировано {n_docs} документов в 'rag_product_context'")

d:\repos\mds25_comm_analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4579.78it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Проиндексировано 15 документов в 'rag_product_context'


In [22]:
all_docs = rag_collection.get(include=["documents", "metadatas"])

for doc_id, meta, doc in zip(
    all_docs["ids"], all_docs["metadatas"], all_docs["documents"]
):
    wc = len(doc.split())
    preview = doc[:80].replace("\n", " ")
    print(f"[{meta['type']}] {doc_id}")
    print(f"  product_id={meta['product_id']} | {wc} слов | {preview}...")
    print()

[product_description] video_platform_description
  product_id=video_platform | 213 слов | Видео-платформа — это онлайн-кинотеатр и ТВ-сервис, где пользователь            ...

[product_description] internet_shop_description
  product_id=internet_shop | 220 слов | Маркетплейс МТС — это сайт, где пользователь ищет и покупает             технику...

[category] video_platform_auth
  product_id=video_platform | 75 слов | Проблемы со входом в приложение Жалобы на невозможность войти в аккаунт и открыт...

[category] video_platform_playback
  product_id=video_platform | 71 слов | Проблемы с воспроизведением или запуском фильмов/каналов Проблема связана с тем,...

[category] video_platform_interface
  product_id=video_platform | 66 слов | Неудобный интерфейс/плеер Жалобы на неудобный интерфейс и плеер Видео-платформа ...

[category] video_platform_choice
  product_id=video_platform | 67 слов | Ассортимент фильмов/каналов Проблема связана с ассортиментом контента в Видео-пл...

[category] video_

## Модуль поиска

1. Берем описание продукта  по ID
2. Делаем семантический запрос по ключевым словам кластера, фильтруем только категории данного продукта.
3. Возвращает строки для подстановки в промпт.

In [ ]:
def retrieve_rag_context(
    collection,
    query: str,
    product_id: str,
    top_k: int = TOP_K_CATEGORIES,
) -> dict:
    """
    Функция возвращает словарь:
        product_description - текст описания продукта
        categories - list[dict] (name, description, similarity)
        rag_context - готовый блок текста для промпта
        categories_str - только названия через '; ' (для инструкции промпта)
    """
    prod_result = collection.get(
        ids=[f"{product_id}_description"],
        include=["documents", "metadatas"],
    )
    product_description = (
        prod_result["documents"][0] if prod_result["documents"] else ""
    )

    available = collection.get(
        where={
            "$and": [
                {"product_id": {"$eq": product_id}},
                {"type": {"$eq": "category"}},
            ]
        },
        include=["documents"],
    )
    actual_k = min(top_k, len(available["ids"]))

    categories = []
    if actual_k > 0:
        cat_results = collection.query(
            query_texts=[query],
            n_results=actual_k,
            where={
                "$and": [
                    {"product_id": {"$eq": product_id}},
                    {"type": {"$eq": "category"}},
                ]
            },
            include=["documents", "metadatas", "distances"],
        )
        for doc, meta, dist in zip(
            cat_results["documents"][0],
            cat_results["metadatas"][0],
            cat_results["distances"][0],
        ):
            categories.append({
                "name": meta["name"],
                "description": doc,
                "similarity": round(1.0 - dist, 4),
            })

    categories_str = "; ".join(c["name"] for c in categories)

    lines = [f"Продукт: {product_description}\n"]
    if categories:
        lines.append("Наиболее релевантные известные категории жалоб:")
        for c in categories:
            lines.append(f"  - {c['name']}: {c['description']}")
    rag_context = "\n".join(lines)

    return {
        "product_description": product_description,
        "categories": categories,
        "rag_context": rag_context,
        "categories_str": categories_str,
    }

In [28]:
test_queries = {
    "video_platform": [
        "Я отключала подписку!!! Почему деньги сейчас списались!",
        "Реклама в приложении жутко тормозит работу самого приложения.",
    ],
    "internet_shop": [
        "Очень долгая выдача предзаказов",
        "Я оплачивал кэшбэком одну модель телефона, а мне подсунули другой  - бэушный, более дешёвый.",
    ],
}

for product_id, queries in test_queries.items():
    print(f"ПРОДУКТ: {product_id}")
    for q in queries:
        result = retrieve_rag_context(rag_collection, q, product_id)
        print(f"Запрос: «{q}»")
        for cat in result["categories"]:
            print(f"{cat['similarity']:.3f}  {cat['name']}")
        print()

ПРОДУКТ: video_platform
Запрос: «Я отключала подписку!!! Почему деньги сейчас списались!»
0.839  Дорогая подписка, сложно отключить подписку
0.782  При переходах между разделами приложение тормозит
0.781  Проблемы со входом в приложение

Запрос: «Реклама в приложении жутко тормозит работу самого приложения.»
0.832  При переходах между разделами приложение тормозит
0.822  Навязчивая реклама
0.811  Проблемы со входом в приложение

ПРОДУКТ: internet_shop
Запрос: «Очень долгая выдача предзаказов»
0.811  Сложный процесс оформления заказа
0.794  Плохо работает сайт, медленно загружается
0.793  Много отвлекающих элементов на сайте

Запрос: «Я оплачивал кэшбэком одну модель телефона, а мне подсунули другой  - бэушный, более дешёвый.»
0.771  Не устроило состояние товара при получении
0.763  Не нашел нужный товар
0.758  Сложный процесс оформления заказа



## Промпты и функция разметки

| Функция | Режим | Контекст |
|---------|-------|----------|
| `build_prompt_no_context` | `no_context` | Только кластер |
| `build_prompt_static` | `static` | Кластер + плоский список названий категорий |
| `build_prompt_rag` | `rag` | Кластер + описание продукта и топ-три категорий с описаниями |

In [29]:
SYS_SHORT = "Ты - аналитик клиентского опыта в телекоммуникационной компании.\nОтвечай строго в формате JSON. Никакого дополнительного текста, только JSON-объект."

JSON_SCHEMA = '{"topic_name": "...", "description": "...", "existing_category": "... или новая тема", "is_new_topic": true/false, "rationale": "...", "severity": "..."}'


def build_prompt_no_context(info: dict):
    keywords_str = ", ".join(info["keywords"][:8])
    comments_str = "\n".join(f"- {c.strip()}" for c in info["comments"])
    return f"Ниже - группа похожих комментариев клиентов, объединённых автоматической кластеризацией.\nКлючевые слова кластера: {keywords_str}\n\nКомментарии:\n{comments_str}\n\nЗадача:\n1. Дай короткое название этой теме (3-7 слов), понятное продуктовому менеджеру.\n2. Опиши суть проблемы в 1-2 предложениях.\n3. Если это новая, ранее неизвестная проблема — укажи 'новая тема'.\n4. Оцени критичность: высокая / средняя / низкая.\n\nОтветь строго в формате JSON:\n{JSON_SCHEMA}"


def build_prompt_static(info: dict, all_category_names: list[str]):
    keywords_str = ", ".join(info["keywords"][:8])
    comments_str = "\n".join(f"- {c.strip()}" for c in info["comments"])
    cats_str = "; ".join(all_category_names)
    return f"Ниже - группа похожих комментариев клиентов, объединённых автоматической кластеризацией.\nКлючевые слова кластера: {keywords_str}\n\nКомментарии:\n{comments_str}\n\nЗадача:\n1. Дай короткое название этой теме (3-7 слов), понятное продуктовому менеджеру.\n2. Опиши суть проблемы в 1-2 предложениях.\n3. Сопоставь тему с одной из существующих категорий ТОЛЬКО при прямом тематическом\n   совпадении (та же причина и явление): {cats_str}.\n   Если точного совпадения нет - обязательно укажи 'новая тема' и объясни в rationale.\n4. Оцени критичность: высокая / средняя / низкая.\n\nОтветь строго в формате JSON:\n{JSON_SCHEMA}"


def build_prompt_rag(info: dict, rag_result: dict) :
    keywords_str = ", ".join(info["keywords"][:8])
    comments_str = "\n".join(f"- {c.strip()}" for c in info["comments"])
    categories_str = rag_result["categories_str"]
    return f"Контекст продукта:\n{rag_result['rag_context']}\n\n---\nНиже — группа похожих комментариев клиентов, объединённых автоматической кластеризацией.\nКлючевые слова кластера: {keywords_str}\n\nКомментарии:\n{comments_str}\n\nЗадача:\n1. Дай короткое название этой теме (3-7 слов), понятное продуктовому менеджеру.\n2. Опиши суть проблемы в 1-2 предложениях.\n3. Сопоставь тему с одной из приведённых выше известных категорий ТОЛЬКО при прямом\n   тематическом совпадении: {categories_str}.\n   Если точного совпадения нет — обязательно укажи 'новая тема' и объясни в rationale,\n   чем она отличается от ближайшей существующей категории.\n4. Оцени критичность: высокая / средняя / низкая.\n\nОтветь строго в формате JSON:\n{JSON_SCHEMA}"


In [31]:
def get_all_category_names(categories_docs: dict, product_id: str) -> list[str]:
    return [cat["name"] for cat in categories_docs.get(product_id, [])]

def label_cluster(info: dict, mode: str, collection=None, categories_docs: dict | None = None):
    product_id = info.get("product_id", "unknown")
    rag_result = None

    if mode == "no_context":
        prompt = build_prompt_no_context(info)

    elif mode == "static":
        cats = get_all_category_names(categories_docs if categories_docs else CATEGORIES_DOCS, product_id)
        prompt = build_prompt_static(info, cats)

    elif mode == "rag":
        if collection is None:
            raise ValueError("mode='rag' требует collection")
        query = ", ".join(info["keywords"][:8])
        rag_result = retrieve_rag_context(collection, query, product_id)
        prompt = build_prompt_rag(info, rag_result)

    else:
        raise ValueError(f"Неизвестный mode: {mode!r}")

    response = call_ollama(prompt, system_prompt=SYS_SHORT)
    parsed = parse_json_response(response["content"])

    if parsed is None:
        parsed = {
            "topic_name": "PARSE_ERROR",
            "description": response["content"][:120],
            "existing_category": "PARSE_ERROR",
            "is_new_topic": None,
            "rationale": "",
            "severity": "",
        }

    ec = str(parsed.get("existing_category", "")).lower()
    if parsed.get("is_new_topic") is None:
        parsed["is_new_topic"] = "новая" in ec or "new" in ec

    return {
        **parsed,
        "mode": mode,
        "topic_id": info["topic_id"],
        "product": info.get("product", ""),
        "product_id": product_id,
        "cluster_size": info.get("size", 0),
        "elapsed": response["elapsed"],
        "tokens": response["tokens"],
        "raw": response["content"],
        "rag_similarities": (
            str([{"name": c["name"], "sim": c["similarity"]}
                 for c in rag_result["categories"]])
            if rag_result else ""
        ),
    }

## Загрузка данных

In [32]:
df_video_platform = pd.read_csv(DATA_DIR / "comments_with_topics_видео_платформа.csv", sep=';')
with open(DATA_DIR / "topic_keywords_видео_платформа.json", encoding="utf-8") as f:
    kw_video_platform = json.load(f)

df_shop = pd.read_csv(DATA_DIR / "comments_with_topics_маркетплейс.csv", sep=';')
with open(DATA_DIR / "topic_keywords_маркетплейс.json", encoding="utf-8") as f:
    kw_shop = json.load(f)

In [35]:
N_CLUSTERS_PER_PRODUCT = 4

def get_top_clusters(df: pd.DataFrame | None, keywords_dict: dict, n: int = 4, exclude_noise: bool = True) -> list[dict]:
    if df is None:
        return []
    counts = df["topic"].value_counts()
    if exclude_noise:
        counts = counts[counts.index != -1]
    top_ids = counts.head(n).index.tolist()
    return [build_cluster_info(df, tid, keywords_dict) for tid in top_ids]

clusters_video_platform = get_top_clusters(df_video_platform, kw_video_platform, N_CLUSTERS_PER_PRODUCT)
clusters_shop = get_top_clusters(df_shop, kw_shop, N_CLUSTERS_PER_PRODUCT)
all_clusters  = clusters_video_platform + clusters_shop

print(f"Кластеров для сравнения: {len(all_clusters)} ({len(clusters_video_platform)} Видео-платформа + {len(clusters_shop)} магазин)\n")
for c in all_clusters:
    print(f"Тема: {c['topic_id']} | {c['product']} | Размер: {c['size']} | Ключевые слова: {', '.join(c['keywords'][:4])}")


Кластеров для сравнения: 8 (4 Видео-платформа + 4 магазин)

Тема: 0 | Видео-платформа | Размер: 256 | Ключевые слова: приложение, подписку, деньги, смотреть
Тема: 1 | Видео-платформа | Размер: 66 | Ключевые слова: фильмов, мало, мало фильмов, фильмы
Тема: 2 | Видео-платформа | Размер: 59 | Ключевые слова: рекламы, реклама, много, много рекламы
Тема: 3 | Видео-платформа | Размер: 39 | Ключевые слова: работает, неудобный, меню, интерфейс
Тема: 0 | Маркетплейс | Размер: 142 | Ключевые слова: доставки, самовывоз, наличии, товара
Тема: 1 | Маркетплейс | Размер: 35 | Ключевые слова: товар, сразу, продаже, через
Тема: 2 | Маркетплейс | Размер: 28 | Ключевые слова: заказ, товар, сказали, деньги
Тема: 3 | Маркетплейс | Размер: 16 | Ключевые слова: цены, выше, очень, даже


## Эксперимент

In [36]:
results = []

for mode in ["no_context", "static", "rag"]:
    print(f"Режим: {mode}")

    for i, cluster_info in enumerate(all_clusters):
        try:
            result = label_cluster(
                info=cluster_info,
                mode=mode,
                collection=rag_collection if mode == "rag" else None,
                categories_docs=CATEGORIES_DOCS,
            )
            results.append(result)
            print(f"[{i + 1}/{len(all_clusters)}] Тема: {cluster_info['topic_id']} ({cluster_info['product']}) -> '{result.get('topic_name', '?')}' | Новая тема: {result.get('is_new_topic')} | {result['elapsed']} с.")
        except Exception as e:
            print(f"ОШИБКА topic={cluster_info['topic_id']}: {e}")

df_results = pd.DataFrame(results)

Режим: no_context
[1/8] Тема: 0 (Видео-платформа) -> 'Проблемы с приложением и подпиской' | Новая тема: False | 17.35 с.
[2/8] Тема: 1 (Видео-платформа) -> 'Проблемы с доступом к фильмам и сериалам' | Новая тема: False | 4.25 с.
[3/8] Тема: 2 (Видео-платформа) -> 'Реклама и трафик' | Новая тема: True | 3.99 с.
[4/8] Тема: 3 (Видео-платформа) -> 'Проблемы с интерфейсом и работоспособностью' | Новая тема: False | 4.47 с.
[5/8] Тема: 0 (Маркетплейс) -> 'Проблемы с доставкой и наличии товара' | Новая тема: False | 4.4 с.
[6/8] Тема: 1 (Маркетплейс) -> 'Проблемы с наличием товара' | Новая тема: True | 4.45 с.
[7/8] Тема: 2 (Маркетплейс) -> 'Проблемы с заказом и доставкой товара' | Новая тема: False | 4.67 с.
[8/8] Тема: 3 (Маркетплейс) -> 'Цены и доставка' | Новая тема: False | 4.56 с.
Режим: static
[1/8] Тема: 0 (Видео-платформа) -> 'Проблемы с приложением и подпиской' | Новая тема: False | 5.08 с.
[2/8] Тема: 1 (Видео-платформа) -> 'Недостаток контента' | Новая тема: False | 3.87 с.
[3/8]

In [37]:
wide = df_results.pivot_table(
    index=["topic_id", "product", "cluster_size"],
    columns="mode",
    values=["topic_name", "existing_category", "is_new_topic", "severity"],
    aggfunc="first",
)

wide.columns = ["_".join(c) for c in wide.columns]
wide = wide[[c for c in sorted(wide.columns)]]

wide.reset_index()

,topic_id,product,cluster_size,existing_category_no_context,existing_category_rag,existing_category_static,is_new_topic_no_context,is_new_topic_rag,is_new_topic_static,severity_no_context,severity_rag,severity_static,topic_name_no_context,topic_name_rag,topic_name_static
0,0,Видео-платформа,256,Приложение и подписка,новая тема,Проблемы с воспроизведением или запуском фильм...,False,True,False,высокая,средняя,высокая,Проблемы с приложением и подпиской,Сложность использования приложения,Проблемы с приложением и подпиской
1,0,Маркетплейс,142,Доставка и наличие товара,новая тема,Не нашел нужный товар,False,True,False,высокая,средняя,средняя,Проблемы с доставкой и наличии товара,Проблемы с доставкой и наличии товара,Проблемы с доставкой и наличием товара
2,1,Видео-платформа,66,Сервис контента,Ассортимент фильмов/каналов,Ассортимент фильмов/каналов,False,False,False,средняя,средняя,средняя,Проблемы с доступом к фильмам и сериалам,Малый ассортимент фильмов и сериалов,Недостаток контента
3,1,Маркетплейс,35,новая тема,новая тема,Не устроило состояние товара при получении,True,True,False,высокая,средняя,средняя,Проблемы с наличием товара,Проблемы с товаром в корзине,Проблемы с товаром в корзине
4,2,Видео-платформа,59,новая тема,PARSE_ERROR,Навязчивая реклама,True,False,False,средняя,,высокая,Реклама и трафик,PARSE_ERROR,Навязчивая реклама и тормоза
5,2,Маркетплейс,28,Проблемы с качеством товара и услуг,Сложный процесс оформления заказа,Не устроило состояние товара при получении,False,False,False,высокая,высокая,высокая,Проблемы с заказом и доставкой товара,Проблемы с заказом и доставкой,Проблемы с заказом и качеством товара
6,3,Видео-платформа,39,Проблемы с интерфейсом и работоспособностью,Проблемы со входом в приложение,Неудобный интерфейс/плеер,False,False,False,высокая,средняя,средняя,Проблемы с интерфейсом и работоспособностью,Проблемы со входом в приложение,Неудобный интерфейс и проблемы с работой сайта
7,3,Маркетплейс,16,Цены и доставка,новая тема,Не устроила цена на товар,False,True,False,средняя,средняя,высокая,Цены и доставка,Цены и доставка,Цены и доставка


In [39]:
print("Среднее время генерации (сек.):")
print(df_results.groupby("mode")["elapsed"].mean().round(2).to_string())
print()

print("Доля 'новая тема' по режимам:")
print(df_results.groupby("mode")["is_new_topic"].mean().round(3).to_string())
print()

print("Кластеры с расхождением в is_new_topic:")
pivot_new = df_results.pivot_table(
    index=["topic_id", "product"],
    columns="mode",
    values="is_new_topic",
    aggfunc="first",
)
disagreements = pivot_new[pivot_new.nunique(axis=1) > 1]
if len(disagreements) > 0:
    print(disagreements.to_string())
else:
    print("Расхождений нет - все три режима совпали")
print()

print("Названия тем по режимам:")
name_pivot = df_results.pivot_table(
    index=["topic_id", "product"],
    columns="mode",
    values="topic_name",
    aggfunc="first",
).reset_index()
for _, row in name_pivot.iterrows():
    print(f"\n  topic={row['topic_id']} | {row['product']}")
    for m in ["no_context", "static", "rag"]:
        print(f"{m}: {row.get(m, 'N/A')}")

Среднее время генерации (сек.):
mode
no_context    6.02
rag           4.88
static        4.16

Доля 'новая тема' по режимам:
mode
no_context    0.25
rag           0.50
static        0.00

Кластеры с расхождением в is_new_topic:
mode                      no_context    rag  static
topic_id product                                   
0        Видео-платформа       False   True   False
         Маркетплейс           False   True   False
1        Маркетплейс            True   True   False
2        Видео-платформа        True  False   False
3        Маркетплейс           False   True   False

Названия тем по режимам:

  topic=0 | Видео-платформа
no_context: Проблемы с приложением и подпиской
static: Проблемы с приложением и подпиской
rag: Сложность использования приложения

  topic=0 | Маркетплейс
no_context: Проблемы с доставкой и наличии товара
static: Проблемы с доставкой и наличием товара
rag: Проблемы с доставкой и наличии товара

  topic=1 | Видео-платформа
no_context: Проблемы с доступ

In [41]:
print("Оценка косинусной близости")

for cluster_info in all_clusters:
    query = ", ".join(cluster_info["keywords"][:8])
    product_id = cluster_info["product_id"]
    rag_result = retrieve_rag_context(rag_collection, query, product_id)

    rag_row = df_results[
    (df_results["topic_id"] == cluster_info["topic_id"]) &
    (df_results["product_id"] == cluster_info["product_id"]) &
    (df_results["mode"] == "rag")
]
    is_new = rag_row["is_new_topic"].iloc[0] if len(rag_row) > 0 else "?"

    print(f"Тема: {cluster_info['topic_id']} ({cluster_info['product']}) | Новая тема: {is_new}")
    print(f"Запрос: {query}")
    for cat in rag_result["categories"]:
        print(f"{cat['similarity']} {cat['name']}")
    print()

Оценка косинусной близости
Тема: 0 (Видео-платформа) | Новая тема: True
Запрос: приложение, подписку, деньги, смотреть, просто, очень, подписки, фильмов
0.8328 Дорогая подписка, сложно отключить подписку
0.8054 Навязчивая реклама
0.801 При переходах между разделами приложение тормозит

Тема: 1 (Видео-платформа) | Новая тема: False
Запрос: фильмов, мало, мало фильмов, фильмы, сериалов, фильмов сериалов, новых, новых фильмов
0.8275 Ассортимент фильмов/каналов
0.8014 Проблемы с воспроизведением или запуском фильмов/каналов
0.7923 При переходах между разделами приложение тормозит

Тема: 2 (Видео-платформа) | Новая тема: False
Запрос: рекламы, реклама, много, много рекламы, часто, трафик, долго, зависает
0.8485 При переходах между разделами приложение тормозит
0.837 Навязчивая реклама
0.8226 Дорогая подписка, сложно отключить подписку

Тема: 3 (Видео-платформа) | Новая тема: False
Запрос: работает, неудобный, меню, интерфейс, неудобное меню, хуйня, поиск, удобное
0.8278 Неудобный интерфейс/

In [42]:
out_path = DATA_DIR / "rag_comparison.csv"
df_results.drop(columns=["raw"], errors="ignore").to_csv(
    out_path, index=False, encoding="utf-8-sig"
)

## Выводы

### Сравнение трёх режимов

| Режим | Поведение | Проблема |
|-------|-----------|----------|
| `no_context` | Придумывает несуществующие категории; везде `is_new_topic=False` | Галлюцинации |
| `static` | Выбирает ближайшую категорию даже при слабом совпадении | Ложные совпадения |
| `rag` | Сопоставляет известные темы; говорит «новая тема» при отсутствии совпадения | — |

Без контекста LLM изобретает правдоподобные, но несуществующие категории  и помечает
все кластеры как `is_new_topic=False`. Из-за этого режим без контекста не пригоден
для задачи обнаружения новых тем, что критично для проекта.

RAG правильно выявил три новые темы в комментариях к Маркетплейсу:
проблемы с доставкой/самовывозом, корзина и каталог, ценообразование.
`static` же сделал ложные совпадения.

Все значения косинусной близости в диапазоне 0.78–0.84. Он слишком узкий для определения новой темы только по одному числу. Детектирование новых тем в пайплайне следует оставить на LLM через поле `rationale`, а не через порог similarity.